# Libraries

In [1]:
from numpy import sin, cos, exp, pi as π
import numpy as np
from plotly import graph_objects as go
import pandas as pd
from sklearn.metrics import mean_squared_error as mse, r2_score as r2

# Analytical model

In [2]:
def model_noise(θ, α, β, Pt=1.0):
    # 1. Divide alpha by 2
    a = α / 2.0
    
    # 2. Probability assignment without dynamic re-scaling
    if Pt == 1.0:
        P_HH = P_HV = P_VH = P_VV = 0.0
    else:
        P_HH = 0.0030
        P_HV = 0.0138
        P_VH = 0.0278  # Using the corrected value
        P_VV = 0.0447
    
    # Pre-compute trigonometric values
    C_a = cos(a)
    S_a = sin(a)
    C_ab = cos(a + β)
    S_ab = sin(a + β)
    C_theta = cos(θ)
    
    # Term 1: Pt * (1/2) * [...]
    term1 = (Pt / 2.0) * (
        C_a**4 + S_a**4 + 2 * (S_a**2) * (C_a**2) * C_theta +
        C_ab**4 + S_ab**4 + 2 * (S_ab**2) * (C_ab**2) * C_theta
    )
    
    # Term 2: ((P_HH + P_VV)/2) * [...]
    term2 = ((P_HH + P_VV) / 2.0) * (
        C_a**4 + S_a**4 + C_ab**4 + S_ab**4
    )
    
    # Term 3: (P_HV + P_VH) * [...]
    term3 = (P_HV + P_VH) * (
        (C_a**2) * (S_a**2) + (C_ab**2) * (S_ab**2)
    )
    
    return term1 + term2 + term3

In [3]:
def ϵ(y_true:np.ndarray, y_pred:np.ndarray):
    y_true_flat = y_true.flatten()
    y_pred_flat = y_pred.flatten()
    # Manually compute epsilon: Sum((y_i - y_hat_i)^2) / Sum((y_i - y_mean)^2)
    sum_squared_residuals = np.sum((y_true_flat - y_pred_flat) ** 2)
    total_sum_squares = np.sum((y_true_flat - np.mean(y_true_flat)) ** 2)
    return sum_squared_residuals / total_sum_squares

# Load Data

In [4]:
df_shifted_filenames = pd.read_csv("/home/qadrian/PHD/QKD/Simulation Results/E91_modified-20260427T224201Z-3-001/E91/shifted_filenames.csv").sort_values(by="alpha").reset_index(drop=True)
#df_shifted_filenames = pd.read_csv("/home/qadrian/QD-Biexciton-cascade-and-polarization-orientation-effect-on-QKD-protocol-E91/Results and Plots/Paper/Fig_6/shifted_mixed_filenames.csv").sort_values(by="alpha").reset_index(drop=True)
df_shifted_filenames

,filename,alpha
0,data_aer_simulator_12_04_2024_14_56_shift_0.csv,0.00
1,data_aer_simulator_12_04_2024_15_00_shift_pi_4...,0.25
2,data_aer_simulator_12_04_2024_14_53_shift_pi_2...,0.50
3,data_aer_simulator_12_04_2024_14_40_shift_3_pi...,0.75
4,data_aer_simulator_19_04_2024_17_04_shift_pi_m...,1.00
5,data_aer_simulator_19_04_2024_17_07_shift_5_pi...,1.25


In [5]:
filenames = df_shifted_filenames["filename"].values
alpha = df_shifted_filenames["alpha"].values*π
subplot_letter = ['f','a','b','c','d','e',]

In [12]:
def plotter(file: str, α: np.float64, index:str, plot_analytical: bool = True):
    #path = "/home/qadrian/QD-Biexciton-cascade-and-polarization-orientation-effect-on-QKD-protocol-E91/Results and Plots/Paper/Fig_6/"
    path = "/home/qadrian/PHD/QKD/Simulation Results/E91_modified-20260427T224201Z-3-001/E91/Resultados/"
    simulation_df = pd.read_csv(path + file, header=None).dropna(axis=1, how='any').values
    dim_theta, dim_beta = simulation_df.shape
    theta = np.linspace(0, 2*np.pi, dim_theta)
    beta = np.linspace(0, np.pi, dim_beta)
    
    analytical_df = np.array([[model_noise(θ=θ, β=β, α=α, Pt=0.9107) for θ in theta] for β in beta], dtype=np.float64).T

    r2_val = r2(y_true=analytical_df.flatten(), y_pred=simulation_df.flatten())
    ϵ_val = ϵ(y_true=analytical_df, y_pred=simulation_df)
    
    fig = go.Figure()
    
    tick_font_size = 15
    axes_label_font_size = 30
    colorbar_tick_fontsize = 18
    title_fontsize = 30
    
    font_family= "Arial"
    
    # Conditionally plot the analytical model
    if plot_analytical:
        fig.add_trace(go.Surface(
            y=theta,
            x=beta, 
            z=analytical_df,
            colorscale='Plasma',
            opacity=1,
            name='Model',
            colorbar=dict(
                x=0.85,
                lenmode='fraction',
                len=0.9,
                tickfont=dict(size=tick_font_size, family=font_family),
                # --- ADDED: Custom ticks to the colorbar ---
                tickmode='array',
                tickvals=[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.89, 1.0],
                ticktext=['0.1','0.2', '0.3' '0.4', '0.5', '0.6', '0.7', '0.8', '0.89 = SL', '1.0']
            )
        ))
    
    # Simulation plot
    fig.add_trace(go.Surface(
        y=theta,
        x=beta, 
        z=simulation_df,
        colorscale='Plasma', 
        opacity=0.8 if plot_analytical else 1.0,
        name='Simulation',
        # --- FIX 1: Applied the same colorbar formatting to the simulation trace ---
        colorbar=dict(
            x=0.95,
            lenmode='fraction',
            len=0.9,
            tickfont=dict(size=colorbar_tick_fontsize, family=font_family),
            # --- ADDED: Custom ticks to the colorbar ---
            tickmode='array',
            tickvals=[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.89, 1.0],
            ticktext=['0.1','0.2', '0.3', '0.4', '0.5', '0.6', '0.7', '0.8', '0.89 = SL', '1.0']
        )
    ))
    
    # --- MODIFIED TRACE: Added showscale=False to eliminate the color bar ---
    fig.add_trace(go.Surface(
        z=0.89 * np.ones(shape=(len(theta), len(beta))), 
        x=beta, 
        y=theta, 
        opacity=0.5,
        colorscale='gray',
        showscale=False 
    ))
    
    alpha_val = (α / np.pi)/2
    a_num, a_den = alpha_val.as_integer_ratio()
    
    if a_num == 0:
        a_str = "0"
        a_str_file = "0"
    elif a_num == 1:
        if a_den ==1:
            a_str = f"π"
            a_str_file = f"pi"
        else:
            a_str = f"π/{a_den}"
            a_str_file = f"pi_{a_den}"
    else:
        if a_den ==1:
            a_str = f"π"
            a_str_file = f"pi"
        else:
            a_str = f"{a_num}π/{a_den}"
            a_str_file = f"{a_num}pi_{a_den}"
    
    fig.update_layout(
        title=dict(
                text=f"({index})\t\tα = {a_str},   ϵ={ϵ_val:1.3e}",
                font=dict(size=title_fontsize, family=font_family),
                x=0.1, y=0.85),
        autosize=False,
        width=750,
        height=750,
        
        scene=dict(
            zaxis=dict(
                title='',
                range=[0, 1.1],
                # --- REVERTED: Back to standard numeric ticks for a cleaner Z-axis ---
                tickvals=[0.2, 0.4, 0.6, 0.8, 1.0], 
                tickfont=dict(size=tick_font_size, family=font_family) 
            ),
            
            yaxis=dict(
                # --- FIX 2: Added <br> before the text to push it away from the numbers ---
                title=dict(
                    text='θ',
                    font=dict(size=axes_label_font_size, family=font_family)), 
                tickfont=dict(size=tick_font_size, family=font_family),                   
                tickmode='array',
                tickvals=[0, np.pi/3, 2*np.pi/3, np.pi, 4*np.pi/3, 5*np.pi/3, 2*np.pi],
                ticktext=['0', 'π/3', '2π/3', 'π', '4π/3', '5π/3', '2π'],
                range=[0, 2 * np.pi]
            ),
            
            xaxis=dict(
                # --- FIX 2: Added <br> before the text to push it away from the numbers ---
                title=dict(
                    text='β',#text='β',
                    font=dict(size=axes_label_font_size, family=font_family)), 
                tickfont=dict(size=tick_font_size, family=font_family),                   
                tickmode='array',
                tickvals=[0, np.pi/4, np.pi/2, 3*np.pi/4, np.pi],
                ticktext=['0', 'π/4', 'π/2', '3π/4', 'π'],
                range=[0, np.pi]
            )
        ),
        
        scene_camera=dict(
            eye=dict(x=-1.6, y=-1.6, z=1.0) 
        )
    )
    
    fig_filename=f"{index}P_erf_corr_alpha_{a_str_file}_eps_{ϵ_val:1.3e}.pdf"
    fig.write_image("/home/qadrian/QD-Biexciton-cascade-and-polarization-orientation-effect-on-QKD-protocol-E91/Results and Plots/Paper/Fig_5/"+fig_filename)
    #fig.write_image(path+fig_filename)
    fig.show()

In [13]:
for k,letter in enumerate(subplot_letter):
    plotter(file=filenames[k],α=alpha[k], index=letter, plot_analytical=False)

/tmp/ipykernel_26487/214497514.py:145: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




/tmp/ipykernel_26487/214497514.py:145: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




/tmp/ipykernel_26487/214497514.py:145: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




/tmp/ipykernel_26487/214497514.py:145: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




/tmp/ipykernel_26487/214497514.py:145: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




/tmp/ipykernel_26487/214497514.py:145: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).


